# RAG Quality Evaluation with DeepEval

This notebook evaluates the RAG pipeline's answer quality using DeepEval metrics.
It calls the RAG agent through LlamaStack, then scores the responses using
LLM-as-a-judge via the vLLM endpoint.

**Metrics evaluated:**
- Answer Relevancy
- Faithfulness (grounding)
- Contextual Relevancy

**Source:** Adapted from [rh-ai-quickstart/RAG evaluations](https://github.com/rh-ai-quickstart/RAG/tree/main/evaluations)

In [ ]:
# Install dependencies (run once)
!pip install -q deepeval llama-stack-client openai

In [ ]:
import os
import json

# Configuration
LLAMASTACK_URL = os.getenv("LLAMASTACK_URL", "http://lsd-rag-service.private-ai.svc.cluster.local:8321")
VLLM_URL = os.getenv("VLLM_URL", "http://granite-8b-agent-predictor.private-ai.svc.cluster.local:8080/v1")
MODEL_ID = "granite-8b-agent"

# Set OPENAI env vars for DeepEval (points to our vLLM)
os.environ["OPENAI_API_KEY"] = "dummy"  # vLLM doesn't need a real key
os.environ["OPENAI_API_BASE"] = VLLM_URL

print(f"LlamaStack: {LLAMASTACK_URL}")
print(f"vLLM: {VLLM_URL}")

In [ ]:
from llama_stack_client import LlamaStackClient, Agent

client = LlamaStackClient(base_url=LLAMASTACK_URL, timeout=120.0)

# Discover vector stores
stores = client.vector_stores.list()
store_map = {}
for vs in stores.data:
    store_map[vs.name] = vs.id
    print(f"  {vs.name}: {vs.id} ({vs.file_counts.completed} files)")

In [ ]:
def ask_rag(question: str, collection: str = "acme_corporate") -> dict:
    """Ask a question to the RAG agent and return answer + retrieved context."""
    store_id = store_map.get(collection, collection)
    
    agent = Agent(
        client,
        model=MODEL_ID,
        instructions="Answer questions based on the provided documents. Be precise and factual.",
        tools=[{
            "name": "builtin::rag/knowledge_search",
            "args": {"vector_store_ids": [store_id]},
        }],
    )
    
    session = agent.create_session(session_name="eval")
    response = agent.create_turn(
        messages=[{"role": "user", "content": question}],
        session_id=session,
        stream=False,
    )
    
    answer = ""
    retrieval_context = []
    
    if hasattr(response, "output_message"):
        answer = response.output_message.content or ""
    
    if hasattr(response, "steps"):
        for step in response.steps or []:
            if hasattr(step, "tool_responses"):
                for tr in step.tool_responses or []:
                    if hasattr(tr, "content") and tr.content:
                        retrieval_context.append(str(tr.content)[:1000])
    
    return {
        "answer": answer if isinstance(answer, str) else str(answer),
        "retrieval_context": retrieval_context,
    }

In [ ]:
# Test cases per scenario
test_cases = {
    "acme_corporate": [
        {"question": "What are the key lithography calibration procedures?",
         "expected": "Calibration involves alignment verification, dose calibration, focus optimization, and overlay measurement."},
        {"question": "What is the maintenance schedule for the UV exposure system?",
         "expected": "UV exposure systems require quarterly lamp replacement, monthly intensity measurements, and weekly uniformity checks."},
        {"question": "What are the safety protocols for chemical handling?",
         "expected": "Chemical handling requires PPE, fume hoods, spill containment, and MSDS documentation."},
    ],
    "eu_ai_act": [
        {"question": "What are the main risk categories in the EU AI Act?",
         "expected": "The EU AI Act defines four risk categories: unacceptable risk, high risk, limited risk, and minimal risk."},
        {"question": "What obligations do providers of high-risk AI systems have?",
         "expected": "Providers must implement risk management, ensure data quality, maintain documentation, enable human oversight, and ensure accuracy."},
    ],
    "whoami": [
        {"question": "What is this person's professional background?",
         "expected": "Based on the CV document provided."},
    ],
}
print(f"Total test cases: {sum(len(v) for v in test_cases.values())}")

In [ ]:
# Run all tests through the RAG agent
results = []
for collection, tests in test_cases.items():
    print(f"\n=== {collection} ===")
    for test in tests:
        print(f"  Q: {test['question'][:60]}...")
        try:
            resp = ask_rag(test["question"], collection)
            print(f"  A: {resp['answer'][:80]}...")
            print(f"  Context chunks: {len(resp['retrieval_context'])}")
            results.append({
                "collection": collection,
                "question": test["question"],
                "expected": test["expected"],
                "answer": resp["answer"],
                "context": resp["retrieval_context"],
            })
        except Exception as e:
            print(f"  ERROR: {e}")
            results.append({
                "collection": collection,
                "question": test["question"],
                "expected": test["expected"],
                "answer": f"ERROR: {e}",
                "context": [],
            })
print(f"\nCollected {len(results)} results")

In [ ]:
from deepeval.models import DeepEvalBaseLLM
from openai import OpenAI

class VLLMJudge(DeepEvalBaseLLM):
    """Uses our vLLM endpoint as the judge model for DeepEval metrics."""
    
    def __init__(self):
        self.client = OpenAI(base_url=VLLM_URL, api_key="dummy")
        self.model_name = MODEL_ID
    
    def load_model(self):
        return self.model_name
    
    def generate(self, prompt: str, **kwargs) -> str:
        resp = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1024,
            temperature=0.0,
        )
        return resp.choices[0].message.content
    
    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)
    
    def get_model_name(self) -> str:
        return self.model_name

judge = VLLMJudge()
print(f"Judge model: {judge.get_model_name()}")

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
)

# Build DeepEval test cases
test_case_objects = []
for r in results:
    tc = LLMTestCase(
        input=r["question"],
        actual_output=r["answer"],
        expected_output=r["expected"],
        retrieval_context=r["context"] if r["context"] else ["No context retrieved"],
    )
    test_case_objects.append(tc)

# Define metrics with our vLLM judge
metrics = [
    AnswerRelevancyMetric(model=judge, threshold=0.5),
    FaithfulnessMetric(model=judge, threshold=0.5),
    ContextualRelevancyMetric(model=judge, threshold=0.5),
]

print(f"Test cases: {len(test_case_objects)}")
print(f"Metrics: {[m.__class__.__name__ for m in metrics]}")

In [ ]:
# Run evaluation
from deepeval.evaluate import evaluate

eval_results = evaluate(
    test_cases=test_case_objects,
    metrics=metrics,
)

In [ ]:
# Display results summary
print("\n" + "=" * 80)
print("RAG Evaluation Results")
print("=" * 80)

for i, (tc, r) in enumerate(zip(test_case_objects, results)):
    print(f"\n[{i+1}] {r['collection']} | Q: {r['question'][:50]}...")
    for metric in metrics:
        try:
            score = metric.score
            passed = '✓' if metric.is_successful() else '✗'
            print(f"    {metric.__class__.__name__}: {score:.2f} {passed}")
        except Exception:
            print(f"    {metric.__class__.__name__}: N/A")

print("\n" + "=" * 80)